# 🌍 DTM Point Cloud Tile Generator — Local Preprocessing
### Run this notebook **LOCALLY** on Ubuntu 24.04 (i7 8th Gen, 16 GB RAM)
> **Purpose:** Read raw LAS/LAZ files → tile into 50 m blocks → generate CSF ground labels → zip for Kaggle upload  
> **Output:** `training_data.zip` (~10–40 MB) ready to upload as a Kaggle Dataset  
> **Do NOT run in Kaggle or Colab** — raw LAS files live on your SSD only

---

```
TWO-MACHINE HYBRID PIPELINE
════════════════════════════════════════════════════════════════════════

YOUR UBUNTU LAPTOP (this notebook, VS Code / JupyterLab)
┌──────────────────────────────────────────────────────────────────┐
│  /home/.../Data/                                                 │
│   ├── Gujrat_Point_Cloud/*.las                                   │
│   ├── Punjab_Point_Cloud/*.las                                   │
│   ├── Rajasthan_Point_Cloud/*.LAS                                │
│   └── ...                                                        │
│                                                                  │
│  [Step 1] CONFIG  →  set your paths below                        │
│  [Step 2] DISCOVER  → scan LAS files                            │
│  [Step 3] PREPROCESS → tile into 50 m .npy blocks               │
│  [Step 4] LABEL  →  CSF pseudo-labels (ground / non-ground)     │
│  [Step 5] QUALITY CHECK → inspect class balance                 │
│  [Step 6] PACKAGE  →  zip → ready to upload to Kaggle           │
└──────────────────────────────────────────────────────────────────┘
                          │  upload .zip (~10–40 MB)
                          ▼
KAGGLE FREE GPU  (dtm_training_kaggle.ipynb — separate notebook)
┌──────────────────────────────────────────────────────────────────┐
│  Train PointNet++ MSG  →  Focal Loss + SWA + OneCycleLR         │
│  Target: >95% accuracy on Indian village terrain                 │
│  Output: best_model.pth + optimal_threshold.json                │
└──────────────────────────────────────────────────────────────────┘
                          │  download weights (< 5 MB)
                          ▼
YOUR UBUNTU LAPTOP (dtm_generation_pipeline.ipynb)
┌──────────────────────────────────────────────────────────────────┐
│  Inference → IDW DTM → Hydrological conditioning                │
│  Output: dtm_conditioned.tif + drainage_streams.gpkg            │
└──────────────────────────────────────────────────────────────────┘
```


In [1]:
# ─── Cell 1: Install local dependencies ──────────────────────────────────────
# Run once. Restart kernel after installation.
import subprocess, sys

pkgs = [
    "laspy[lazrs,laszip]==2.5.4",
    "numpy>=1.24",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "tqdm",
    "pathlib",
]

# cloth-simulation-filter is the Python CSF binding
csf_pkgs = ["cloth-simulation-filter"]

print("Installing core packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)

print("Installing CSF (cloth simulation filter)...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + csf_pkgs)
    HAS_CSF = True
    print("✅ CSF installed")
except Exception:
    HAS_CSF = False
    print("⚠️  CSF not installed — will use fallback height-threshold labeller")

print("\n✅ All dependencies ready. Restart kernel if this is the first run.")


Installing core packages...
Installing CSF (cloth simulation filter)...
✅ CSF installed

✅ All dependencies ready. Restart kernel if this is the first run.


In [2]:
# ─── Cell 2: Configuration ─────────────────────────────────────────────────────
from pathlib import Path
import os
from collections import defaultdict

# ══════════════════════════════════════════════════════════════════════════════
#  ▶▶  CONFIGURATION - EDIT ONLY THIS BLOCK  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

CONFIG = {
    # Paths
    "data_root": "/home/jaideepchouhan/Documents/AIR Docs/GSI 2026/Data/Zip files",
    "output_root": "/home/jaideepchouhan/Documents/AIR Docs/GSI 2026/Preprocessed",
    "upload_zip": "/home/jaideepchouhan/Documents/AIR Docs/GSI 2026/colab_upload/training_data.zip",
    
    # Tiling parameters
    "block_size_m": 50,
    "stride_m": 40,
    "min_pts_per_block": 200,
    "max_pts_per_block": 4096,
    "outlier_pct": (1.0, 99.0),
    
    # CSF ground filtering
    "csf_cloth_resolution": 0.5,
    "csf_class_threshold": 0.3,
    "csf_iterations": 500,
    
    # Data split
    "train_val_split": 0.80,
    "random_seed": 42,
    
    # Processing
    "cpu_threads": 8,
}

# ══════════════════════════════════════════════════════════════════════════════
#  🚀 SETUP & VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

def setup_directories():
    """Create all necessary directories"""
    Path(CONFIG["output_root"]).mkdir(parents=True, exist_ok=True)
    Path(CONFIG["upload_zip"]).parent.mkdir(parents=True, exist_ok=True)
    (Path(CONFIG["output_root"]) / "train").mkdir(exist_ok=True)
    (Path(CONFIG["output_root"]) / "val").mkdir(exist_ok=True)

def find_las_files(root_path):
    """Find all LAS/LAZ files recursively"""
    las_files = []
    extensions = {'.las', '.laz'}
    
    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            if Path(filename).suffix.lower() in extensions:
                las_files.append(Path(dirpath) / filename)
    return las_files

def extract_location(file_path, data_root):
    """Extract state and village from file path"""
    rel = file_path.relative_to(data_root)
    parts = rel.parts
    
    if len(parts) >= 2:
        state = parts[0].replace('_', ' ').strip()
        village = parts[1].replace('_', ' ').strip()
    else:
        state, village = "unknown", file_path.stem
    
    return state, village

# ══════════════════════════════════════════════════════════════════════════════
#  📊 EXECUTION
# ══════════════════════════════════════════════════════════════════════════════

# Setup directories
setup_directories()

# Find all LAS/LAZ files
data_root = Path(CONFIG["data_root"])
las_files = find_las_files(data_root)

# Group files by location
location_groups = defaultdict(list)
for f in las_files:
    state, village = extract_location(f, data_root)
    location_groups[(state, village)].append(f)

# Print summary
print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Data root:     {CONFIG['data_root']}")
print(f"Output root:   {CONFIG['output_root']}")
print(f"Upload zip:    {CONFIG['upload_zip']}")
print(f"\nTiling:        {CONFIG['block_size_m']}m blocks, {CONFIG['stride_m']}m stride")
print(f"Ground filter: CSF (res={CONFIG['csf_cloth_resolution']}m, thresh={CONFIG['csf_class_threshold']}m)")
print(f"Train/val:     {CONFIG['train_val_split']*100:.0f}/{100-CONFIG['train_val_split']*100:.0f}")
print(f"CPU threads:   {CONFIG['cpu_threads']}")
print("=" * 60)

print(f"\n📁 Found {len(las_files)} LAS/LAZ files")
for (state, village), files in location_groups.items():
    print(f"   • {state} / {village}: {len(files)} file(s)")

print("\n✅ Ready for processing")

CONFIGURATION SUMMARY
Data root:     /home/jaideepchouhan/Documents/AIR Docs/GSI 2026/Data/Zip files
Output root:   /home/jaideepchouhan/Documents/AIR Docs/GSI 2026/Preprocessed
Upload zip:    /home/jaideepchouhan/Documents/AIR Docs/GSI 2026/colab_upload/training_data.zip

Tiling:        50m blocks, 40m stride
Ground filter: CSF (res=0.5m, thresh=0.3m)
Train/val:     80/20
CPU threads:   8

📁 Found 10 LAS/LAZ files
   • Andaman and Nicobar Islands 1 / Gandhinagar Diglipur group1 densified point cloud.laz: 1 file(s)
   • Punjab Point Cloud / DHUNDA FATEHGARH SAHIB 32619.laz: 1 file(s)
   • Punjab Point Cloud / Dhal Hoshiarpur 31235.las: 1 file(s)
   • Andaman and Nicobar Islands 2 / Kadamtala Rangat A&N 02022022 group1 densified point cloud.laz: 1 file(s)
   • Gujrat Point Cloud / DEVDI POINT CLOUD (511671).las: 1 file(s)
   • Gujrat Point Cloud / KHAPRETA 510206.laz: 1 file(s)
   • Tamil Nadu Point Cloud / PIRAYANKUPPAM.las: 1 file(s)
   • Tamil Nadu Point Cloud / THANDALAM.las: 1 file

In [3]:
# ─── Cell 3: Discover LAS/LAZ files ──────────────────────────────────────────
import os
from pathlib import Path
from collections import defaultdict

DATA_ROOT = Path(CONFIG["data_root"])

# Recursive search — handles nested folders like Tamil Nadu's structure
LAS_EXTENSIONS = {".las", ".laz", ".LAS", ".LAZ"}
all_files = [p for p in DATA_ROOT.rglob("*") if p.suffix in LAS_EXTENSIONS]

# Group by immediate village sub-folder
village_files = defaultdict(list)
for p in all_files:
    # The village is the first folder level under DATA_ROOT
    try:
        rel = p.relative_to(DATA_ROOT)
        village = rel.parts[0]
    except Exception:
        village = "unknown"
    village_files[village].append(p)

print(f"\n📂 Discovered {len(all_files)} LAS/LAZ files across {len(village_files)} village folders")
print()

total_est_gb = 0
for village, files in sorted(village_files.items()):
    sz = sum(f.stat().st_size for f in files if f.exists()) / 1e9
    total_est_gb += sz
    print(f"  {village:<50s}  {len(files):4d} files  {sz:.2f} GB")

print(f"\n  {'TOTAL':<50s}  {len(all_files):4d} files  {total_est_gb:.2f} GB")
print()
print("✅ Discovery complete — all files are LOCAL (never leave your machine)")

# Save list for next cell
ALL_LAS_FILES  = all_files
VILLAGE_FILES  = dict(village_files)



📂 Discovered 10 LAS/LAZ files across 6 village folders

  Andaman and Nicobar Islands 1                          1 files  2.04 GB
  Andaman and Nicobar Islands 2                          1 files  11.60 GB
  Gujrat_Point_Cloud                                     2 files  3.49 GB
  Punjab_Point_Cloud                                     2 files  2.32 GB
  Rajasthan_Point_Cloud                                  2 files  1.93 GB
  Tamil Nadu_Point_Cloud                                 2 files  10.38 GB

  TOTAL                                                 10 files  31.76 GB

✅ Discovery complete — all files are LOCAL (never leave your machine)


In [4]:
# ─── Cell 4: Inspect first LAS file ──────────────────────────────────────────
import laspy, numpy as np

if not ALL_LAS_FILES:
    print("❌ No LAS files found. Check CONFIG['data_root']")
else:
    sample = ALL_LAS_FILES[0]
    print(f"Peeking at: {sample.name}")
    with laspy.open(str(sample)) as f:
        hdr = f.header
        print(f"  LAS version   : {hdr.version}")
        print(f"  Point count   : {hdr.point_count:,}")
        print(f"  Point format  : {hdr.point_format.id}")
        has_rgb = any(d.name in {'red','green','blue'} for d in hdr.point_format.dimensions)
        has_int = 'intensity' in [d.name for d in hdr.point_format.dimensions]
        print(f"  Has RGB       : {has_rgb}")
        print(f"  Has Intensity : {has_int}")
        print(f"  Scale         : {hdr.scales}")
        print(f"  Offset        : {hdr.offsets}")
        print(f"  X range       : {hdr.x_min:.2f} – {hdr.x_max:.2f}  ({hdr.x_max-hdr.x_min:.0f} m)")
        print(f"  Y range       : {hdr.y_min:.2f} – {hdr.y_max:.2f}  ({hdr.y_max-hdr.y_min:.0f} m)")
        print(f"  Z range       : {hdr.z_min:.2f} – {hdr.z_max:.2f}  ({hdr.z_max-hdr.z_min:.1f} m)")


Peeking at: Gandhinagar_Diglipur_group1_densified_point_cloud.laz
  LAS version   : 1.2
  Point count   : 287,661,850
  Point format  : 3
  Has RGB       : True
  Has Intensity : True
  Scale         : [0.001 0.001 0.001]
  Offset        : [ 502000. 1476000.       0.]
  X range       : 500229.95 – 503963.66  (3734 m)
  Y range       : 1475319.91 – 1478177.05  (2857 m)
  Z range       : -24.65 – 93.96  (118.6 m)


In [5]:
# ─── Cell 5: Preprocessing engine — LAS → tiled .npy blocks ──────────────────
#
# WHY THIS DESIGN:
#   PointNet++ processes fixed-size point clouds (4096 pts).
#   A single LAS file may contain millions of points — we tile it into
#   overlapping 50 m × 50 m blocks, each subsampled to ≤4096 points.
#   Overlap (stride < block_size) ensures border points appear in multiple
#   tiles so the model sees complete local context for every point.
# ─────────────────────────────────────────────────────────────────────────────

import laspy
import numpy as np
from pathlib import Path

def load_las(path: Path) -> np.ndarray:
    """Load XYZ from LAS/LAZ. Returns (N,3) float32."""
    with laspy.open(str(path)) as f:
        las = f.read()
    xyz = np.stack([
        np.asarray(las.x, dtype=np.float64),
        np.asarray(las.y, dtype=np.float64),
        np.asarray(las.z, dtype=np.float64),
    ], axis=1)
    return xyz.astype(np.float32)


def clip_outliers(xyz: np.ndarray, pct=(1.0, 99.0)) -> np.ndarray:
    """Remove Z outliers (birds, noise returns) by percentile clipping."""
    z_lo, z_hi = np.percentile(xyz[:, 2], pct)
    mask = (xyz[:, 2] >= z_lo) & (xyz[:, 2] <= z_hi)
    return xyz[mask]


def tile_point_cloud(xyz, block_size, stride, min_pts, max_pts, rng):
    """
    Slide a 2-D grid over XY and yield overlapping blocks.
    Each block is normalised to a local coordinate frame (origin = block min).
    Sub-samples to max_pts using FPS-like stratified sampling when block is large.
    """
    x_min, y_min = xyz[:, 0].min(), xyz[:, 1].min()
    x_max, y_max = xyz[:, 0].max(), xyz[:, 1].max()

    x_starts = np.arange(x_min, x_max, stride)
    y_starts = np.arange(y_min, y_max, stride)

    tiles = []
    for xs in x_starts:
        for ys in y_starts:
            mask = (
                (xyz[:, 0] >= xs) & (xyz[:, 0] < xs + block_size) &
                (xyz[:, 1] >= ys) & (xyz[:, 1] < ys + block_size)
            )
            blk = xyz[mask]
            if len(blk) < min_pts:
                continue

            # Subsample
            if len(blk) > max_pts:
                idx = rng.choice(len(blk), size=max_pts, replace=False)
                blk = blk[idx]

            # Normalise to block origin (local frame)
            blk_norm = blk.copy()
            blk_norm[:, 0] -= xs
            blk_norm[:, 1] -= ys
            # Z: keep absolute Z for terrain features, subtract median to centre
            blk_norm[:, 2] -= np.median(blk_norm[:, 2])

            tiles.append((blk_norm.astype(np.float32), mask))
    return tiles


print("✅ Preprocessing engine ready")


✅ Preprocessing engine ready


In [6]:
# ─── Cell 6: CSF Label Generator ─────────────────────────────────────────────
#
# Cloth Simulation Filter (CSF): Zhang et al., 2016.
# Simulates a rigid cloth draped from above onto an inverted point cloud.
# Points within csf_class_threshold metres of the cloth → ground (1).
# Points above the threshold → non-ground (0).
#
# Fallback (if CSF not installed): height-percentile heuristic.
# The fallback is less accurate but good enough for pseudo-label generation.
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np

try:
    import CSF
    HAS_CSF = True
except ImportError:
    HAS_CSF = False


def label_with_csf(xyz: np.ndarray, cfg: dict) -> np.ndarray:
    """
    Returns binary label array (N,) — 1 = ground, 0 = non-ground.
    Uses CSF if available, otherwise height-percentile fallback.
    """
    if HAS_CSF:
        csf = CSF.CSF()
        csf.params.bSloopSmooth          = True
        csf.params.cloth_resolution      = cfg["csf_cloth_resolution"]
        csf.params.class_threshold       = cfg["csf_class_threshold"]
        csf.params.interations           = cfg["csf_iterations"]
        csf.setPointCloud(xyz.tolist())

        ground_idx     = CSF.VecInt()
        non_ground_idx = CSF.VecInt()
        csf.do_filtering(ground_idx, non_ground_idx)

        labels = np.zeros(len(xyz), dtype=np.int64)
        labels[list(ground_idx)] = 1
        return labels
    else:
        # ── Fallback: progressive height-threshold ─────────────────────────
        # Approximate cloth by computing local minimum Z in grid cells.
        # Points within 0.4 m of local minimum → ground.
        xyz64 = xyz.astype(np.float64)
        x_min, y_min = xyz64[:, 0].min(), xyz64[:, 1].min()
        x_rng = xyz64[:, 0].max() - x_min + 1e-6
        y_rng = xyz64[:, 1].max() - y_min + 1e-6

        GW = int(np.clip(x_rng / 1.0, 16, 64))
        GH = int(np.clip(y_rng / 1.0, 16, 64))

        xi = np.clip(((xyz64[:, 0] - x_min) / x_rng * GW).astype(int), 0, GW-1)
        yi = np.clip(((xyz64[:, 1] - y_min) / y_rng * GH).astype(int), 0, GH-1)
        ci = yi * GW + xi

        c_min = np.full(GH * GW, np.inf)
        np.minimum.at(c_min, ci, xyz64[:, 2])
        empty = np.isinf(c_min)
        c_min[empty] = xyz64[:, 2].mean()

        height_above = xyz64[:, 2] - c_min[ci]
        labels = (height_above <= cfg["csf_class_threshold"]).astype(np.int64)
        return labels


def validate_labels(labels: np.ndarray, tile_name: str) -> bool:
    """
    Reject tiles with degenerate label distribution.
    A good tile has 5%–80% ground points.
    """
    n = len(labels)
    if n == 0:
        return False
    gnd_pct = labels.sum() / n
    if gnd_pct < 0.05 or gnd_pct > 0.80:
        # Edge case: pure vegetation canopy or pure open field
        # Keep open fields (all ground) but warn; reject pure-canopy tiles
        if gnd_pct > 0.80:
            return True   # open agricultural field — include it
        return False
    return True


print(f"✅ CSF labeller ready  (CSF library: {HAS_CSF})")
print(f"   Using {'CSF cloth simulation' if HAS_CSF else 'height-threshold fallback'}")


✅ CSF labeller ready  (CSF library: True)
   Using CSF cloth simulation


In [ ]:
# ─── Cell 7: Main Processing Loop ────────────────────────────────────────────
# This is the heavy cell — reads every LAS file, tiles it, labels it, saves .npy
# ⏱  Estimated time: 5–30 min depending on total data size
# ─────────────────────────────────────────────────────────────────────────────

import time, json
from tqdm import tqdm

rng = np.random.default_rng(CONFIG["random_seed"])

OUT = Path(CONFIG["output_root"])
TRAIN_DIR = OUT / "train"
VAL_DIR   = OUT / "val"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)

stats = {
    "total_las_files": len(ALL_LAS_FILES),
    "total_tiles_written": 0,
    "train_tiles": 0,
    "val_tiles": 0,
    "skipped_sparse": 0,
    "skipped_degenerate_labels": 0,
    "class_dist": [0, 0],          # [non-ground, ground]
}

tile_counter = 0
file_bar = tqdm(ALL_LAS_FILES, desc="LAS files", unit="file")

for las_path in file_bar:
    file_bar.set_postfix({"tiles": tile_counter, "file": las_path.name[:30]})
    try:
        xyz = load_las(las_path)
    except Exception as e:
        print(f"  ⚠️  Skip {las_path.name}: {e}")
        continue

    # Remove Z outliers
    xyz = clip_outliers(xyz, CONFIG["outlier_pct"])
    if len(xyz) < CONFIG["min_pts_per_block"]:
        continue

    # Generate tiles
    tiles = tile_point_cloud(
        xyz,
        block_size = CONFIG["block_size_m"],
        stride     = CONFIG["stride_m"],
        min_pts    = CONFIG["min_pts_per_block"],
        max_pts    = CONFIG["max_pts_per_block"],
        rng        = rng,
    )

    for blk_xyz, _ in tiles:
        # Generate labels
        labels = label_with_csf(blk_xyz, CONFIG)

        if not validate_labels(labels, f"tile_{tile_counter:06d}"):
            stats["skipped_degenerate_labels"] += 1
            continue

        # Train / val split
        is_val = rng.random() > CONFIG["train_val_split"]
        split_dir = VAL_DIR if is_val else TRAIN_DIR
        tile_name = f"tile_{tile_counter:06d}"
        dest = split_dir / tile_name
        dest.mkdir(exist_ok=True)

        np.save(dest / "points.npy", blk_xyz)
        np.save(dest / "labels.npy", labels.astype(np.int8))

        stats["class_dist"][0] += int((labels == 0).sum())
        stats["class_dist"][1] += int((labels == 1).sum())

        if is_val:
            stats["val_tiles"] += 1
        else:
            stats["train_tiles"] += 1
        stats["total_tiles_written"] += 1
        tile_counter += 1

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  ✅ PREPROCESSING COMPLETE")
print(f"{'═'*60}")
print(f"  LAS files processed     : {stats['total_las_files']}")
print(f"  Total tiles written     : {stats['total_tiles_written']}")
print(f"    → Train               : {stats['train_tiles']}")
print(f"    → Val                 : {stats['val_tiles']}")
print(f"  Skipped (sparse)        : {stats['skipped_sparse']}")
print(f"  Skipped (bad labels)    : {stats['skipped_degenerate_labels']}")
tot = sum(stats['class_dist'])
if tot > 0:
    gnd_pct = 100 * stats['class_dist'][1] / tot
    print(f"  Ground point fraction   : {gnd_pct:.1f}%  ({stats['class_dist'][1]:,} / {tot:,})")
print(f"{'═'*60}")

# Save stats
with open(OUT / "preprocessing_stats.json", "w") as f:
    json.dump(stats, f, indent=2)
print(f"  Stats saved to: {OUT / 'preprocessing_stats.json'}")


LAS files:   0%|          | 0/10 [00:00<?, ?file/s, tiles=0, file=Gandhinagar_Diglipur_group1_de]

In [ ]:
# ─── Cell 8: Quality Check & Visualisation ───────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import random

OUT = Path(CONFIG["output_root"])
train_tiles = sorted((OUT / "train").glob("tile_*"))
val_tiles   = sorted((OUT / "val").glob("tile_*"))

print(f"Train tiles: {len(train_tiles)}")
print(f"Val tiles  : {len(val_tiles)}")

# ── Check class balance across a sample of 200 tiles ──────────────────────
sample_tiles = random.sample(list(train_tiles), min(200, len(train_tiles)))
gnd_fracs = []
for t in sample_tiles:
    lb = np.load(t / "labels.npy")
    gnd_fracs.append(lb.mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(gnd_fracs, bins=30, color="#2196F3", edgecolor="white")
axes[0].axvline(np.mean(gnd_fracs), color="red", ls="--", label=f"Mean {np.mean(gnd_fracs):.2f}")
axes[0].set_title("Ground Point Fraction per Tile (train sample)")
axes[0].set_xlabel("Fraction of ground points")
axes[0].set_ylabel("Tile count")
axes[0].legend()

# ── Visualise one random tile ──────────────────────────────────────────────
t = random.choice(sample_tiles)
xyz = np.load(t / "points.npy")
lbl = np.load(t / "labels.npy")
ax = axes[1]
sc = ax.scatter(xyz[:, 0], xyz[:, 1], c=lbl, cmap="RdYlGn", s=2, alpha=0.7)
plt.colorbar(sc, ax=ax, label="0=non-ground  1=ground")
ax.set_title(f"Tile: {t.name}  ({len(xyz)} pts,  {lbl.mean()*100:.1f}% ground)")
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(str(OUT / "quality_check.png"), dpi=120)
plt.show()
print(f"\n✅ Quality check plot saved: {OUT}/quality_check.png")


In [ ]:
# ─── Cell 9: Package Training Data for Kaggle Upload ─────────────────────────
#
# WHY ZIP AND NOT UPLOAD FOLDER DIRECTLY?
#   Kaggle's dataset uploader accepts a zip archive.
#   The .npy files are tiny (30–80 KB each) so even 18,000 tiles
#   compress to a 10–40 MB zip — fast to upload.
#   Raw LAS files (100 MB–2 GB each) NEVER leave your machine.
# ─────────────────────────────────────────────────────────────────────────────

import zipfile, time
from pathlib import Path
from tqdm import tqdm

OUT      = Path(CONFIG["output_root"])
ZIP_DEST = Path(CONFIG["upload_zip"])
ZIP_DEST.parent.mkdir(parents=True, exist_ok=True)

# Collect all tile files
all_npy = list(OUT.rglob("*.npy"))
print(f"Compressing {len(all_npy)} .npy files into {ZIP_DEST.name} ...")

t0 = time.time()
with zipfile.ZipFile(str(ZIP_DEST), "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for npy in tqdm(all_npy, desc="Zipping", unit="file"):
        arcname = npy.relative_to(OUT)    # preserves train/val structure
        zf.write(str(npy), str(arcname))

elapsed = time.time() - t0
size_mb = ZIP_DEST.stat().st_size / 1e6
print(f"\n✅ ZIP created in {elapsed:.1f}s")
print(f"   Path    : {ZIP_DEST}")
print(f"   Size    : {size_mb:.1f} MB")
print()
print("NEXT STEPS:")
print("  1. Go to  kaggle.com → Datasets → New Dataset")
print("  2. Upload: ", ZIP_DEST)
print("  3. Name it 'point-cloud-data-of-10-indian-villages' (match notebook reference)")
print("  4. Open dtm_training_kaggle.ipynb in Kaggle → Add this dataset → Run all")
